# Обучение модели предсказания кредитного дефолта

Этот ноутбук содержит процесс подготовки данных, обучения и валидации GRU-модели для предсказания риска дефолта клиентов.

In [1]:
import sys
from pathlib import Path

__ROOT__ = Path.cwd()
if not (__ROOT__ / "credit_risk_agent").exists() and (__ROOT__.parent / "credit_risk_agent").exists():
    __ROOT__ = __ROOT__.parent

if str(__ROOT__) not in sys.path:
    sys.path.append(str(__ROOT__))

from credit_risk_agent.config import (
    DATABASE_PATH,
    ID_COL,
    SCALER_COLS,
    SCALER_PATH,
    TARGET_COL,
)

## 1. Загрузка библиотек и модулей проекта

In [2]:
import sqlite3
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Импорт собственных модулей проекта
from credit_risk_agent.model.model import CreditDefaultPredictor
from credit_risk_agent.model.dataset import prepare_dataset
from credit_risk_agent.data.preprocessing import preprocess
from credit_risk_agent.data.normalization import normalize, fit_and_save_scaler

## 2. Загрузка и предобработка данных (ETL)

Загружаем данные из SQLite базы данных, объединяем информацию о клиентах и истории платежей, производим базовую чистку признаков и делим выборку на обучающую (train) и тестовую (test) с сохранением баланса классов (стратификация).

In [3]:
with sqlite3.connect(DATABASE_PATH) as conn:
    client_df = pd.read_sql_query("SELECT * FROM clients", conn)
    history_df = pd.read_sql_query("SELECT * FROM payment_history", conn)

df = pd.merge(client_df, history_df, on="client_id")
df = preprocess(df)

client_data = df[[ID_COL, TARGET_COL]].drop_duplicates()
train_ids, test_ids = train_test_split(
    client_data[ID_COL],
    test_size=0.2,
    stratify=client_data[TARGET_COL],
    random_state=42
)

train_df = df[df[ID_COL].isin(train_ids)]
test_df = df[df[ID_COL].isin(test_ids)]

fit_and_save_scaler(train_df, SCALER_COLS, SCALER_PATH)
train_df = normalize(train_df, SCALER_COLS, SCALER_PATH)
test_df = normalize(test_df, SCALER_COLS, SCALER_PATH)

## 3. Подготовка датасетов для PyTorch

Используем общую функцию `prepare_dataset` из модуля `model.dataset` для преобразования временных последовательностей клиентов в 3D тензоры (формат `[batch_size, seq_len, num_features]`) и создания `CreditDataset`.

In [4]:
train_dataset = prepare_dataset(train_df, id_col=ID_COL, target_col=TARGET_COL)
test_dataset = prepare_dataset(test_df, id_col=ID_COL, target_col=TARGET_COL)

/home/iraedeus/Projects/credit-risk-agent/credit_risk_agent/model/dataset.py:20: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.sequences = torch.as_tensor(sequences, dtype=torch.float32)


## 4. Обучение модели

Инициализируем GRU-модель, лосс-функцию (с учетом дисбаланса классов) и оптимизатор Adam.  
В качестве лосс-функции используем `BCEWithLogitsLoss` с параметром `pos_weight`, который задается как отношение негативных примеров к позитивным.

In [5]:
BATCH_SIZE = 64
LEARNING_RATE = 0.001

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = CreditDefaultPredictor(hidden_size=64, num_layers=1, static_size=14, dropout_prob=0.28)

pos_weight = torch.tensor([78.0 / 22.0])
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")
for epoch in range(25):
    model.train()
    epoch_losses = []
    for seq_features, static_features, labels in train_loader:
        seq_features = seq_features.float()
        static_features = static_features.float()
        labels = labels.float()
        
        optimizer.zero_grad()
        outputs = model(seq_features, static_features)
        
        loss_val = loss_fn(outputs, labels)
        
        epoch_losses.append(loss_val.item())
        
        loss_val.backward()
        optimizer.step()
        
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    print(f"Epoch {epoch+1:02d}/25 - Average Loss: {avg_loss:.4f}")

Starting training...


Epoch 01/25 - Average Loss: 0.9384


Epoch 02/25 - Average Loss: 0.9002


Epoch 03/25 - Average Loss: 0.8938


Epoch 04/25 - Average Loss: 0.8899


...

Epoch 21/25 - Average Loss: 0.8660


Epoch 22/25 - Average Loss: 0.8672


Epoch 23/25 - Average Loss: 0.8631


Epoch 24/25 - Average Loss: 0.8624


Epoch 25/25 - Average Loss: 0.8641


## 5. Валидация модели на тестовом наборе данных

Оцениваем качество обученной модели на отложенной тестовой выборке с расчетом метрики ROC AUC и классификационного отчета (Classification Report).

In [6]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for seq_features, static_features, labels in test_loader:
        seq_features = seq_features.float()
        static_features = static_features.float()
        outputs = model(seq_features, static_features)
        
        probs = torch.sigmoid(outputs)
        all_preds.extend(probs.numpy())
        all_targets.extend(labels.numpy())

roc_auc = roc_auc_score(all_targets, all_preds)
print(f"Test ROC AUC: {roc_auc:.4f}\n")

# Бинаризация предсказаний с порогом 0.5 для отчета классификации
binary_preds = [1 if p >= 0.55 else 0 for p in all_preds]
print("Classification Report:")
print(classification_report(all_targets, binary_preds, target_names=["Non-default", "Default"]))

Test ROC AUC: 0.7768

Classification Report:
              precision    recall  f1-score   support

 Non-default       0.87      0.83      0.85      4673
     Default       0.49      0.58      0.53      1327

    accuracy                           0.77      6000
   macro avg       0.68      0.70      0.69      6000
weighted avg       0.79      0.77      0.78      6000

